# Notebook 03: Bronze to Silver Transformations
**Exam Coverage**: Section 3 (Incremental Data Processing)
**Duration**: 60-75 minutes
---
## Learning Objectives
By the end of this notebook, you will be able to:
- Implement data quality transformations (deduplication, standardization)
- Validate and cleanse customer and product data
- Process streaming sales with joins and enrichment
- Implement Slowly Changing Dimension (SCD) Type 2
- Create data quality monitoring dashboards
- Apply business rules and constraints
---

## Section 1: Introduction to Medallion Architecture
The Medallion Architecture organizes data into three layers:
### 🥉 Bronze Layer (Raw)
- **Purpose**: Exact copy of source data
- **Quality**: May contain duplicates, nulls, invalid data
- **Schema**: Raw, unmodified
- **Use case**: Audit trail, reprocessing
### 🥈 Silver Layer (Cleaned)
- **Purpose**: Validated, cleansed, conformed data
- **Quality**: Deduplicated, standardized, validated
- **Schema**: Consistent, business-friendly
- **Use case**: Analytics, ML feature engineering
### 🥇 Gold Layer (Business)
- **Purpose**: Aggregated, business-level metrics
- **Quality**: Highly curated, denormalized
- **Schema**: Optimized for query performance
- **Use case**: BI reports, dashboards, KPIs

**This notebook**: Bronze → Silver transformations

Import shared variables and configuration

In [0]:
%run ./variables

# Configuration Variables

Central configuration file for the Databricks Data Engineer Certification Lab.

**Usage**: Import this file in all notebooks to maintain consistent naming.

```python
%run ./variables
```

## Unity Catalog Configuration

## Volume Paths

## Checkpoint Locations

## Table Names

...

## Browser Types

## Locations (US Cities)

## Helper Functions

## Validation

## Display Configuration Summary

In [0]:
# Set current catalog and schema to Silver
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"USE SCHEMA {SILVER_SCHEMA}")

print(f"Current Catalog: {spark.catalog.currentCatalog()}")
print(f"Current Schema: {spark.catalog.currentDatabase()}")

Current Catalog: cert_prep_catalog
Current Schema: `02_silver`


In [0]:
# Import required functions
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

## Section 2: Clean Customer Data
Customer data typically requires:
- **Deduplication**: Remove duplicate customer records
- **Standardization**: Consistent formats for email, phone, names
- **Validation**: Valid email patterns, phone numbers
- **Null handling**: Business rules for required fields

Let's start by examining the bronze data.

In [0]:
# Read bronze customer data
customers_bronze = spark.table(CUSTOMERS_BRONZE_TABLE)

print(f"Bronze customer count: {customers_bronze.count():,}")
customers_bronze.printSchema()
display(customers_bronze.limit(10))

Bronze customer count: 10,200
root
 |-- customer_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



customer_id,email,first_name,last_name,location,loyalty_tier,phone,registration_date,_rescued_data
f0bd4810-2e5c-472b-8cb0-eb598c23d835,lisa.miller.3825@email.com,Lisa,Miller,"Detroit, MI, USA",Platinum,+1-530-370-7011,2025-10-09,null
df23a90d-142b-413f-b287-de7f3bea989d,patricia.martinez.3826@email.com,Patricia,Martinez,"Columbus, OH, USA",Silver,null,2026-05-03,null
ed31a79d-45f8-4322-8548-55fd68e1a989,emily.martinez.3827@email.com,Emily,Martinez,"Atlanta, GA, USA",Bronze,+1-417-676-2896,2026-03-22,null
1d3ce36e-180e-463a-acae-0b82ed0fec58,emily.rodriguez.3828@email.com,Emily,Rodriguez,"Detroit, MI, USA",Silver,+1-940-428-1215,2024-12-07,null
46f1f835-3749-4da9-a354-3e29d843562a,sarah.garcia.3829@email.com,null,Garcia,"Chicago, IL, USA",Gold,+1-336-571-9228,2024-06-11,null
5464b7a7-1238-47d1-83c3-89e1238dca38,lisa.lopez.3830@email.com,Lisa,Lopez,"San Jose, CA, USA",Bronze,+1-717-948-3017,2025-11-13,null
b7ae8894-f30b-4386-8ddb-0b298de83ebb,jennifer.garcia.3831@email.com,Jennifer,gARCIA,"San Antonio, TX, USA",Bronze,+1-271-403-5253,2025-05-27,null
c7435fb0-3fec-427c-94be-7bd95730d2df,david.garcia.3832@email.com,David,Garcia,"Miami, FL, USA",null,+1-862-614-5864,2024-08-20,null
f29239b9-749a-4038-b517-976cef2acce3,sarah.martinez.3833@email.com,sARAH,Martinez,"Denver, CO, USA",Bronze,+1-869-289-3482,2026-04-19,null
4dde769b-8bbf-4e03-8716-634b6c7c0032,jane.gonzalez.3834@email.com,Jane,Gonzalez,"Fort Worth, TX, USA",Bronze,null,2024-06-16,null


In [0]:
# Data quality assessment
customers_bronze.select(
    F.count("*").alias("total_records"),
    F.count("customer_id").alias("non_null_ids"),
    F.countDistinct("customer_id").alias("unique_ids"),
    F.count("email").alias("non_null_emails"),
    F.count("phone").alias("non_null_phones"),
    F.sum(F.when(F.col("email").isNull() | F.col("phone").isNull(), 1).otherwise(0)).alias("missing_contact")
).show()

+-------------+------------+----------+---------------+---------------+---------------+
|total_records|non_null_ids|unique_ids|non_null_emails|non_null_phones|missing_contact|
+-------------+------------+----------+---------------+---------------+---------------+
|        10200|       10200|     10000|          10200|           8156|           2044|
+-------------+------------+----------+---------------+---------------+---------------+



In [0]:
# Check for duplicate customer_ids
duplicate_check = customers_bronze.groupBy("customer_id").count().filter("count > 1")
duplicate_count = duplicate_check.count()

print(f"Duplicate customer IDs: {duplicate_count}")
if duplicate_count > 0:
    display(duplicate_check.limit(10))

Duplicate customer IDs: 198


customer_id,count
441cb999-c516-4a43-8b5a-3066515e9ee3,2
5ac6df01-8d03-48d0-9ecb-ce7f4537c134,2
d4ce40d7-bab4-47a6-a972-7f61a2ac2dc3,2
be8e5805-cffc-46b1-895e-8ee4e5ff6568,2
3a449975-9be9-47b0-9874-3f31db5d8b6f,2
43033b27-bf8d-4614-a131-6f9baca7230b,2
976dd428-b2b1-4612-89d5-74a652dfee05,2
1136a11a-a383-4976-ae27-0a16b7475c7d,2
c12ca866-abbf-4754-9b02-d905d0a73e66,2
edc662f1-f090-4868-837c-9e7ad35652cc,2


### Deduplication Strategy
When multiple records exist for the same `customer_id`, we need a tiebreaker:
| Strategy | Approach |
|----------|----------|
| **Most recent** | Keep row with latest timestamp |
| **Most complete** | Keep row with fewest nulls |
| **Explicit priority** | Use a priority column |

**Common pattern**: Use window functions with `row_number()`

---
### 🎯 EXERCISE 1: Deduplicate Customer Records
**Your task**: Remove duplicate `customer_id` records using a window function.
**Requirements:**
- Create a window partitioned by `customer_id`
- Order by `customer_id` descending (arbitrary tiebreaker)
- Use `row_number()` to assign numbers
- Keep only rows where `row_num == 1`
- Drop the helper column
- Store result in `customers_deduped`

**Pattern:**
```python
window_spec = Window.partitionBy("col").orderBy(F.col("col").desc())
df.withColumn("row_num", F.row_number().over(window_spec))
```
**Hint**: Import `Window` from `pyspark.sql.window` (already done above).

In [0]:
# TODO: Deduplicate customers using window function

# Create window spec
window_spec = Window.partitionBy("customer_id").orderBy(F.col("customer_id").desc())

# Apply row_number and filter
customers_deduped = customers_bronze \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

# SQL version of the above:
# spark.sql("""
#           CREATE OR REPLACE TEMPORARY VIEW customers_deduped AS
#           SELECT *
#           FROM customers_bronze
#           QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY customer_id DESC) = 1
#           """)
    
print(f"After deduplication: {customers_deduped.count():,}")

After deduplication: 10,000


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Deduplicate Customers

# window_spec = Window.partitionBy("customer_id").orderBy(F.col("customer_id").desc())

# customers_deduped = customers_bronze \
#     .withColumn("row_num", F.row_number().over(window_spec)) \
#     .filter(F.col("row_num") == 1) \
#     .drop("row_num")

# print(f"✅ After deduplication: {customers_deduped.count():,}")

---
### 🎯 EXERCISE 2: Standardize and Validate Customer Data
**Your task**: Clean and add validation flags to customer data.
**Part 1 - Standardization:**
- Email: lowercase and trim
- Phone: remove non-numeric characters
- Names: proper case (initcap) and trim
- Add `processed_at` timestamp
**Part 2 - Validation Flags:**
- `is_valid_email`: Check regex pattern
- `is_valid_phone`: Check length == 10
- `data_quality_score`: Count of valid fields (0-4)

**Functions you'll need:**
```python
F.lower(), F.trim(), F.initcap()
F.regexp_replace(col, "[^0-9]", "")  # Keep only digits
F.col("email").rlike(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")
F.current_timestamp()
```
**Hint**: Build in steps - standardize first, then add flags.

In [0]:
# TODO: Clean and standardize customer data

customers_clean = customers_deduped.select(
    F.col("customer_id"),
    # TODO: Apply initcap and trim to first_name
    F.initcap(F.trim(F.col("first_name"))).alias("first_name"),
    # TODO: Apply initcap and trim to last_name
    F.initcap(F.trim(F.col("last_name"))).alias("last_name"),
    # TODO: Lowercase and trim email
    F.lower(F.trim(F.col("email"))).alias("email"),
    # TODO: Remove non-digits from phone
    F.regexp_replace(F.col("phone"),"[^0-9]", "").alias("phone"),
    F.col("location"),
    F.col("loyalty_tier"),
    F.col("registration_date").alias("account_created_date"),
    # TODO: Add processed_at timestamp
    F.current_timestamp().alias("processed_at")
)

# TODO: Add validation flags
customers_clean = customers_clean.withColumn(
    "is_valid_email",
    # TODO: Add email regex validation
    F.col("email").rlike(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")
).withColumn(
    "is_valid_phone",
    # TODO: Check phone length == 10
    F.length(F.col("phone")) == 10
).withColumn(
    "data_quality_score",
    # TODO: Sum up quality indicators (email not null + phone not null + valid email + valid phone)
    (
        F.when(F.col("email").isNotNull(), 1).otherwise(0) +
        F.when(F.col("phone").isNotNull(), 1).otherwise(0) +
        F.when(F.col("is_valid_email").isNotNull(), 1).otherwise(0) +
        F.when(F.col("is_valid_phone").isNotNull(), 1).otherwise(0) 
    )    
)

# SQL version of the above
# spark.sql("""
#          CREATE OR REPLACE TEMPORARY VIEW customers_clean AS 
#          SELECT customer_id,
#          initcap(trim(first_name)) AS first_name,
#          initcap(trim(last_name)) AS last_name,
#          lower(trim(email)) AS email, 
#          regexp_replace(phone, "[^0-9]", "") AS phone,
#          location,
#          loyalty_tier,
#          registration_date AS account_created_date,
#          current_timestamp() AS processed_at,
#          email RLIKE "^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$" AS is_valid_email,
#          length(regexp_replace(phone,"[^0-9]","")) = 10 AS is_valid_phone,len(phone) = 10 AS is_valid_phone,
#          (
#                CASE WHEN email IS NOT NULL THEN 1 ELSE 0 END +
#                CASE WHEN phone IS NOT NULL THEN 1 ELSE 0 END +
#                CASE WHEN is_valid_email THEN 1 ELSE 0 END +
#                CASE WHEN is_valid_phone THEN 1 ELSE 0 END AS data_quality_score
#          )
#          FROM customers_deduped
#          """)

display(customers_clean.limit(10))

customer_id,first_name,last_name,email,phone,location,loyalty_tier,account_created_date,processed_at,is_valid_email,is_valid_phone,data_quality_score
00100b41-7bd8-418b-ac9d-92eed31d151d,Mary,Davis,mary.davis.9071@email.com,null,"Seattle, WA, USA",Bronze,2024-05-26,2026-05-21T20:04:57.369Z,true,null,2
001622bf-2350-45d5-9d2c-3dc82b471e0b,Jennifer,Johnson,jennifer.johnson.116@email.com,17313902292,"Jacksonville, FL, USA",Silver,2025-09-25,2026-05-21T20:04:57.369Z,true,false,4
0021df65-e8be-4f7d-9cdb-2d17169de9e3,Lisa,Lopez,lisa.lopez.5795@email.com,18262586465,"Phoenix, AZ, USA",Silver,2024-12-10,2026-05-21T20:04:57.369Z,true,false,4
002555c5-c29d-4138-ad0c-a4f9aba31d44,Mary,Davis,mary.davis.957@email.com,17491075610,"Fort Worth, TX, USA",Bronze,2025-11-26,2026-05-21T20:04:57.369Z,true,false,4
0031a3ac-e9db-4d4d-abca-dd7741478832,David,Gonzalez,david.gonzalez.2123@email.com,null,null,Bronze,2024-10-01,2026-05-21T20:04:57.369Z,true,null,2
0033252e-929e-425b-93ec-bf1498e5e200,Michael,Miller,michael.miller.9412@email.com,null,"Indianapolis, IN, USA",null,2024-10-21,2026-05-21T20:04:57.369Z,true,null,2
003792c6-d793-40fe-bcab-b226f10b1263,Lisa,Johnson,lisa.johnson.360@email.com,null,"Chicago, IL, USA",Silver,2026-04-19,2026-05-21T20:04:57.369Z,true,null,2
00403d31-e929-47f5-96bd-6e456f4abc83,Lisa,Gonzalez,lisa.gonzalez.5494@email.com,14729042841,"New York, NY, USA",Silver,2025-07-11,2026-05-21T20:04:57.369Z,true,false,4
0044c5d3-157b-4243-b349-7aa1cd694b0b,Thomas,Davis,thomas.davis.8721@email.com,12502774853,null,null,2026-02-14,2026-05-21T20:04:57.369Z,true,false,4
004765f7-6ce7-436b-b0a7-c498bc03b015,Patricia,Lopez,patricia.lopez.6604@email.com,18751942911,"Dallas, TX, USA",null,2024-12-06,2026-05-21T20:04:57.369Z,true,false,4


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Standardize and Validate

# customers_clean = customers_deduped.select(
#     F.col("customer_id"),
#     F.trim(F.initcap(F.col("first_name"))).alias("first_name"),
#     F.trim(F.initcap(F.col("last_name"))).alias("last_name"),
#     F.lower(F.trim(F.col("email"))).alias("email"),
#     F.regexp_replace(F.col("phone"), "[^0-9]", "").alias("phone"),
#     F.col("location"),
#     F.col("loyalty_tier"),
#     F.col("registration_date").alias("account_created_date"),
#     F.current_timestamp().alias("processed_at")
# )

# customers_clean = customers_clean.withColumn(
#     "is_valid_email",
#     F.col("email").rlike(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")
# ).withColumn(
#     "is_valid_phone",
#     F.length(F.col("phone")) == 10
# ).withColumn(
#     "data_quality_score",
#     (
#         F.when(F.col("email").isNotNull(), 1).otherwise(0) +
#         F.when(F.col("phone").isNotNull(), 1).otherwise(0) +
#         F.when(F.col("is_valid_email"), 1).otherwise(0) +
#         F.when(F.col("is_valid_phone"), 1).otherwise(0)
#     )
# )

# display(customers_clean.limit(10))

---
### Business Rules and Filtering
Now apply business rules to filter out low-quality records.

In [0]:
# Filter out records that don't meet minimum quality standards
# Business rule: Must have valid customer_id and at least one valid contact method
customers_validated = customers_clean.filter(
    (F.col("customer_id").isNotNull()) &
    ((F.col("is_valid_email")) | (F.col("is_valid_phone")))
)

print(f"Records passing validation: {customers_validated.count():,}")
print(f"Records filtered out: {customers_clean.count() - customers_validated.count():,}")

Records passing validation: 9,908
Records filtered out: 92


In [0]:
# Write cleaned customer data to Silver table
customers_validated.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(CUSTOMERS_SILVER_TABLE)

print(f"✅ Created Silver table: {CUSTOMERS_SILVER_TABLE}")

✅ Created Silver table: cert_prep_catalog.02_silver.customers_clean


## Section 3: Clean Product Data
Product data requires similar treatment:
- **Deduplication**: Remove duplicate products
- **Format fixes**: Standardize names, categories, SKUs
- **Price validation**: Ensure positive, reasonable prices
- **Calculated fields**: Profit margins

Let's examine the data first.

In [0]:
# Read bronze product data
products_bronze = spark.table(PRODUCTS_BRONZE_TABLE)

print(f"Bronze product count: {products_bronze.count():,}")
products_bronze.printSchema()
display(products_bronze.limit(10))

Bronze product count: 1,010
root
 |-- category: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- inventory_count: integer (nullable = true)
 |-- last_updated: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



category,cost,inventory_count,last_updated,price,product_id,product_name,subcategory,_rescued_data
Electronics,121.42,164,2026-05-14T17:35:49.000Z,273.77,P-1758,Premium Monitor,Computers,null
Clothing,null,148,2026-05-14T17:35:49.000Z,470.27,P-1759,Deluxe Scarf,null,null
Clothing,696.32,466,2026-05-14T17:35:49.000Z,1321.9,P-1760,Classic Jacket,Kids,null
Sports & Outdoors,93.06,283,2026-05-14T17:35:49.000Z,137.24,P-1761,Performance Yoga Mat,null,null
Books,612.85,117,2026-05-14T17:35:49.000Z,978.58,P-1762,History: A Story,Comics,null
Clothing,801.1,157,2026-05-14T17:35:49.000Z,1534.36,P-1763,Ultimate Scarf,Kids,null
Home & Garden,231.94,466,2026-05-14T17:35:49.000Z,354.37,P-1764,Rustic Mirror,Decor,null
sPORTS & oUTDOORS,206.48,186,2026-05-14T17:35:49.000Z,502.5,P-1765,Ultimate Water Bottle,Team Sports,null
Electronics,613.9,214,2026-05-14T17:35:49.000Z,933.89,P-1766,Premium Tablet,Computers,null
Electronics,null,36,2026-05-14T17:35:49.000Z,334.64,P-1767,Essential Tablet Pro,null,null


In [0]:
# Assess product data quality
products_bronze.select(
    F.count("*").alias("total_records"),
    F.countDistinct("product_id").alias("unique_products"),
    F.count("price").alias("non_null_prices"),
    F.sum(F.when(F.col("price") <= 0, 1).otherwise(0)).alias("invalid_prices"),
    F.count("category").alias("non_null_categories")
).show()

+-------------+---------------+---------------+--------------+-------------------+
|total_records|unique_products|non_null_prices|invalid_prices|non_null_categories|
+-------------+---------------+---------------+--------------+-------------------+
|         1010|           1000|           1010|             7|               1010|
+-------------+---------------+---------------+--------------+-------------------+



---
### 🎯 EXERCISE 3: Clean and Validate Product Data
**Your task**: Apply the same cleaning pattern to products.
**Part 1 - Deduplication:**
- Window partitioned by `product_id`
- Keep first row per product
**Part 2 - Standardization:**
- Trim product_name, category, subcategory
- Cast price and cost to `decimal(10,2)`
- Add `processed_at`
**Part 3 - Validation:**
- `is_valid_price`: price > 0 AND price < 100000
- `is_valid_cost`: cost > 0 AND cost <= price
- `profit_margin`: ((price - cost) / price) * 100
**Part 4 - Filter:**
- Must have product_id, product_name, and valid price

**Hint**: Follow the customer cleaning pattern.

In [0]:
# TODO: Deduplicate products

window_spec = Window.partitionBy("product_id").orderBy(F.col("product_id").desc())

products_deduped = products_bronze \
    .withColumn("row_num", F.row_number().over(window_spec)) \
        .filter(F.col("row_num") == 1 ) \
            .drop("row_num")

# SQL Version of the above

#spark.sql("""
#    CREATE OR REPLACE TEMPORARY VIEW products_deduped AS 
#    SELECT * FROM products_bronze
#    QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY product_id DESC) = 1
#    ;
#    """
#)



print(f"After deduplication: {products_deduped.count():,}")

After deduplication: 1,000


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Deduplicate Products

# window_spec = Window.partitionBy("product_id").orderBy(F.col("product_id").desc())

# products_deduped = products_bronze \
#     .withColumn("row_num", F.row_number().over(window_spec)) \
#     .filter(F.col("row_num") == 1) \
#     .drop("row_num")

# print(f"✅ After deduplication: {products_deduped.count():,}")

In [0]:
# TODO: Clean, standardize, and validate products

products_clean = products_deduped.select(
    F.col("product_id"),
    # TODO: Trim product_name
    F.trim(F.col("product_name")).alias("product_name"),
    # TODO: Trim category
    F.trim(F.col("category")).alias("category"),
    # TODO: Trim subcategory
    F.trim(F.col("subcategory")).alias("subcategory"),
    # TODO: Cast price to decimal(10,2)
    F.col("price").cast("decimal(10,2)").alias("price"),
    # TODO: Cast cost to decimal(10,2)
    F.col("cost").cast("decimal(10,2)").alias("cost"),
    # TODO: Add processed_at
    F.current_timestamp().alias("processed_at")
)

# TODO: Add validation flags
products_clean = products_clean.withColumn(
    "is_valid_price",
    # TODO: price > 0 AND price < 100000
    (F.col("price") > 0) & (F.col("price") < 100000)
).withColumn(
    "is_valid_cost",
    # TODO: cost > 0 AND cost <= price
    (F.col("cost") > 0) & (F.col("cost") <= F.col("price"))
).withColumn(
    "profit_margin",
    # TODO: Calculate (price - cost) / price * 100, handle nulls
    F.when(
        (F.col("price") > 0) & (F.col("cost").isNotNull()),
        ((F.col("price") - F.col("cost")) / F.col("price") * 100)
    ).otherwise(None)
    )

# SQL Version of the above:

#spark.sql("""
#          CREATE OR REPLACE TEMPORARY VIEW products_clean AS
#          SELECT product_id,
#          product_name,
#          category,
#          subcategory,
#          price,
#          cost,
#          processed_at,
#          is_valid_price,
#          is_valid_cost,
#          CASE 
#            WHEN is_valid_price 
#            AND is_valid_cost 
#            THEN ((price - cost) / price) * 100 
#          END AS profit_margin
#          FROM (
#                SELECT 
#                product_id,
#                trim(product_name) AS product_name,
#                trim(category) AS category,
#                trim(subcategory) AS subcategory,
#                CAST(price AS DECIMAL (10,2)) AS price,
#                CAST(cost AS DECIMAL (10,2)) AS cost,
#                current_timestamp() AS processed_at,
#                (price > 0 AND price < 100000) AS is_valid_price,
#                (cost > 0 AND cost <= price) AS is_valid_cost
#                FROM products_deduped
#                ) sub_q
#          """)

display(products_clean.limit(10))

product_id,product_name,category,subcategory,price,cost,processed_at,is_valid_price,is_valid_cost,profit_margin
P-1001,Classic Jacket,Clothing,Men,1031.69,496.99,2026-05-21T20:05:11.300Z,true,true,51.8275838672500
P-1002,The Professional History,Books,Non-Fiction,17.48,9.27,2026-05-21T20:05:11.300Z,true,true,46.9679633867300
P-1003,Rustic Clock,Home & Garden,Furniture,1854.85,764.75,2026-05-21T20:05:11.300Z,true,true,58.7702509636900
P-1004,Smart Keyboard,Electronics,Wearables,517.77,282.55,2026-05-21T20:05:11.300Z,true,true,45.4294377812500
P-1005,Designer Dress,Clothing,Men,1816.65,798.86,2026-05-21T20:05:11.300Z,true,true,56.0256516114800
P-1006,Essential Curtains,Home & Garden,Storage,298.97,188.74,2026-05-21T20:05:11.300Z,true,true,36.8699200588700
P-1007,Professional Chair,Home & Garden,null,1292.22,737.58,2026-05-21T20:05:11.300Z,true,true,42.9214839578400
P-1008,The Premium Science,BOOKS,Comics,210.63,null,2026-05-21T20:05:11.300Z,true,null,null
P-1009,Smart Speaker,Electronics,Audio,780.93,null,2026-05-21T20:05:11.300Z,true,null,null
P-1010,Ultimate Table,Home & Garden,Decor,1248.13,529.12,2026-05-21T20:05:11.300Z,true,true,57.6069800421400


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Clean and Validate Products

# products_clean = products_deduped.select(
#     F.col("product_id"),
#     F.trim(F.col("product_name")).alias("product_name"),
#     F.trim(F.col("category")).alias("category"),
#     F.trim(F.col("subcategory")).alias("subcategory"),
#     F.col("price").cast("decimal(10,2)").alias("price"),
#     F.col("cost").cast("decimal(10,2)").alias("cost"),
#     F.current_timestamp().alias("processed_at")
# )

# products_clean = products_clean.withColumn(
#     "is_valid_price",
#     (F.col("price") > 0) & (F.col("price") < 100000)
# ).withColumn(
#     "is_valid_cost",
#     (F.col("cost") > 0) & (F.col("cost") <= F.col("price"))
# ).withColumn(
#     "profit_margin",
#     F.when(
#         (F.col("price") > 0) & (F.col("cost").isNotNull()),
#         ((F.col("price") - F.col("cost")) / F.col("price") * 100)
#     ).otherwise(None)
# )

# display(products_clean.limit(10))

In [0]:
# Filter products meeting quality standards
products_validated = products_clean.filter(
    (F.col("product_id").isNotNull()) &
    (F.col("product_name").isNotNull()) &
    (F.col("is_valid_price"))
)

print(f"Products passing validation: {products_validated.count():,}")

Products passing validation: 993


In [0]:
# Write to Silver table
products_validated.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(PRODUCTS_SILVER_TABLE)

print(f"✅ Created Silver table: {PRODUCTS_SILVER_TABLE}")

✅ Created Silver table: cert_prep_catalog.02_silver.products_clean


## Section 4: Process Sales with Streaming Joins
Sales transactions are high-volume and require:
- **Streaming processing**: Handle continuous data flow
- **Join enrichment**: Add customer and product details
- **Calculated fields**: Subtotals, validation flags
- **Stream-static joins**: Join streaming sales with static dimensions
### Stream-Static Joins
**Pattern**: Stream (sales) joins static (customers, products)
```python
streaming_df.join(static_df, "key", "left")
```
**Important**: Dimension tables must be small enough to fit in memory.

In [0]:
# Read bronze sales as a stream
sales_bronze_stream = spark.readStream \
    .format("delta") \
    .table(SALES_BRONZE_TABLE)

sales_bronze_stream.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- line_items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- product_id: string (nullable = true)
 |    |    |-- quantity: string (nullable = true)
 |    |    |-- unit_price: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_address: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
from pyspark.sql.functions import explode, col
from pyspark.sql.types import IntegerType, DecimalType

orders_flat = sales_bronze_stream.withColumn("line_item", explode("line_items")) \
    .select(
        col("order_id"),
        col("customer_id"),
        col("order_status"),
        col("order_timestamp"),
        col("payment_method"),
        col("shipping_address"),
        col("line_item.product_id").alias("product_id"),
        col("line_item.quantity").cast(IntegerType()).alias("quantity"),
        col("line_item.unit_price").cast(DecimalType(10,2)).alias("unit_price"),
        col("_rescued_data")
    )

In [0]:
# Load dimension tables (static)
customers_dim = spark.table(CUSTOMERS_SILVER_TABLE)
products_dim = spark.table(PRODUCTS_SILVER_TABLE)

---
### 🎯 EXERCISE 4: Enrich Sales with Dimensions
**Your task**: Join streaming sales with customer and product dimensions.
**Requirements:**
1. Join sales with customers (left join on `customer_id`)
- Select: customer_id, first_name, last_name, loyalty_tier
2. Join result with products (left join on `product_id`)
- Select: product_id, product_name, category, subcategory, price
3. Create final select with:
- All key fields
  - `customer_name`: Concatenate first + last name
  - `subtotal`: quantity * unit_price
- Other relevant fields
  - `is_valid_sale`: Check all required fields exist

**Hint**: Chain the joins, then select columns.

In [0]:
# TODO: Join sales with dimensions and transform

# Join with customers
sales_enriched = orders_flat \
    .join(
        # TODO: Select customer fields and join
        customers_dim.select("customer_id", "first_name", "last_name", "loyalty_tier"),
        "customer_id",
        "left"  
    ) \
    .join(
        # TODO: Select product fields and join
        products_dim.select("product_id", "product_name", "category", "subcategory", "price"),
        "product_id",
        "left"      
    )

# TODO: Select and transform fields
sales_clean = sales_enriched.select(
    F.col("order_id"),
    F.col("customer_id"),
    # TODO: Concatenate customer name
    F.concat(F.col("first_name"), F.lit(" "), F.col("last_name")).alias("customer_name"),
    F.col("product_id"),
    F.col("product_name"),
    F.col("category"),
    F.col("subcategory"),
    F.col("quantity"),
    F.col("unit_price"),
    # TODO: Calculate subtotal
    (F.col("quantity") * F.col("unit_price")).alias("subtotal"),
    F.col("payment_method"),
    F.col("order_timestamp"),
    F.col("loyalty_tier"),
    F.current_timestamp().alias("processed_at")
)

# TODO: Add validation flag
sales_clean = sales_clean.withColumn(
    "is_valid_sale",
    # TODO: Check required fields
    (F.col("customer_id").isNotNull()) &
    (F.col("product_id").isNotNull()) &
    (F.col("quantity") > 0) &
    (F.col("unit_price") > 0)
)

# SQL version of the above:


#spark.sql("""
#         CREATE OR REPLACE TEMPORARY VIEW sales_clean AS
#         SELECT 
#         sales.order_id,
#         sales.customer_id,
#         sales.customer_name,
#         sales.product_id,
#         sales.product_name,
#         sales.category,
#         sales.subcategory,
#         sales.quantity,
#         sales.unit_price,
#         sales.subtotal,
#         sales.payment_method,
#         sales.order_timestamp,
#         sales.loyalty_tier,
#         sales.processed_at,
#         (sales.customer_id IS NOT NULL) AND (sales.product_id IS NOT NULL) AND (sales.quantity > 0) AND       #         (sales.unit_price > 0) AS is_valid_sale
#         FROM 
#            (
#            SELECT 
#            ord.order_id,
#            ord.customer_id,
#            concat(cust.first_name, ' ', cust.last_name) AS customer_name,
#            ord.product_id,
#            prod.product_name,
#            prod.category,
#            prod.subcategory,
#            ord.quantity,
#            ord.unit_price,
#            (ord.quantity * ord.unit_price) AS subtotal,
#            ord.payment_method,
#            ord.order_timestamp,
#            cust.loyalty_tier,
#            current_timestamp() AS processed_at
#            FROM orders_flat ord
#            LEFT JOIN cert_prep_catalog.02_silver.customers_clean cust ON ord.customer_id = cust.customer_id
#            LEFT JOIN cert_prep_catalog.02_silver.products_clean prod ON ord.product_id = prod.product_id
#            ) sales
#          """)

---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Enrich Sales with Joins

# sales_enriched = orders_flat \
#     .join(
#         customers_dim.select("customer_id", "first_name", "last_name", "loyalty_tier"),
#         "customer_id",
#         "left"
#     ) \
#     .join(
#         products_dim.select("product_id", "product_name", "category", "subcategory", "price"),
#         "product_id",
#         "left"
#     )

# sales_clean = sales_enriched.select(
#     F.col("order_id"),
#     F.col("customer_id"),
#     F.concat(F.col("first_name"), F.lit(" "), F.col("last_name")).alias("customer_name"),
#     F.col("product_id"),
#     F.col("product_name"),
#     F.col("category"),
#     F.col("subcategory"),
#     F.col("quantity"),
#     F.col("unit_price"),
#     (F.col("quantity") * F.col("unit_price")).alias("total_amount"),
#     F.col("payment_method"),
#     F.col("order_timestamp"),
#     F.col("loyalty_tier"),
#     F.current_timestamp().alias("processed_at")
# )

# sales_clean = sales_clean.withColumn(
#     "is_valid_sale",
#     (F.col("customer_id").isNotNull()) &
#     (F.col("product_id").isNotNull()) &
#     (F.col("quantity") > 0) &
#     (F.col("total_amount") > 0)
# )

# Production best practice: Add watermark for late-arriving data
# sales_clean = sales_clean.withWatermark("sale_timestamp", "1 hour")
# This allows data up to 1 hour late to be processed correctly

In [0]:
# Write streaming sales to Silver table
sales_query = sales_clean \
    .filter(F.col("is_valid_sale")) \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", SALES_SILVER_CHECKPOINT_PATH) \
    .trigger(availableNow=True) \
    .table(SALES_SILVER_TABLE)

sales_query.awaitTermination()

print(f"✅ Sales data processed to: {SALES_SILVER_TABLE}")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7383001953244502>, line 2
      1 # Write streaming sales to Silver table
----> 2 sales_query = sales_clean \
      3     .filter(F.col("is_valid_sale")) \
      4     .writeStream \
      5     .format("delta") \
      6     .outputMode("append") \
      7     .option("checkpointLocation", SALES_SILVER_CHECKPOINT_PATH) \
      8     .trigger(availableNow=True) \
      9     .table(SALES_SILVER_TABLE)
     11 sales_query.awaitTermination()
     13 print(f"✅ Sales data processed to: {SALES_SILVER_TABLE}")

NameError: name 'sales_clean' is not defined

In [0]:
# Verify sales Silver data
sales_count = spark.table(SALES_SILVER_TABLE).count()
print(f"Total sales in Silver: {sales_count:,}")

display(spark.table(SALES_SILVER_TABLE).limit(10))

## Section 5: Implement SCD Type 2
**Slowly Changing Dimension (SCD) Type 2** tracks historical changes.
### How SCD Type 2 Works
When a dimension record changes:
1. **Close** the existing record (set `effective_end_date`, `is_current = false`)
2. **Insert** a new record with updated values
3. New record has `effective_start_date = today`, `is_current = true`
### SCD Type 2 Columns
| Column | Purpose |
|--------|--------|
| `effective_start_date` | When this version became active |
| `effective_end_date` | When superseded (NULL = current) |
| `is_current` | Boolean flag for current record |
| `version` | Version number (optional) |
### Example
Customer changes loyalty tier from Bronze → Silver:
**Before:**
```
customer_id | tier   | start_date | end_date | is_current
C001       | Bronze | 2024-01-01 | NULL     | true
```
**After:**
```
customer_id | tier   | start_date | end_date   | is_current
C001       | Bronze | 2024-01-01 | 2024-06-01 | false
C001       | Silver | 2024-06-01 | NULL       | true
```

In [0]:
# Check if SCD2 table exists, if not create it
try:
    scd2_table = spark.table(CUSTOMERS_SCD2_TABLE)
    print(f"SCD2 table exists with {scd2_table.count():,} records")
except:
    print("Creating initial SCD2 table...")
    
    # Initialize with current customer data
    initial_scd2 = spark.table(CUSTOMERS_SILVER_TABLE).select(
        F.col("customer_id"),
        F.col("first_name"),
        F.col("last_name"),
        F.col("email"),
        F.col("phone"),
        F.col("location"),
        F.col("loyalty_tier"),
        F.col("account_created_date").cast("date"),
        F.col("account_created_date").cast("date").alias("effective_start_date"),
        F.lit(None).cast("date").alias("effective_end_date"),
        F.lit(True).alias("is_current"),
        F.lit(1).alias("version")
    )
    
    initial_scd2.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(CUSTOMERS_SCD2_TABLE)
    
    print(f"✅ Created {CUSTOMERS_SCD2_TABLE}")

In [0]:
# Display current SCD2 table
display(spark.table(CUSTOMERS_SCD2_TABLE).limit(10))

### Simulate Changes
Let's simulate some loyalty tier changes to demonstrate SCD Type 2.

In [0]:
# Simulate loyalty tier upgrades
updates = spark.table(CUSTOMERS_SILVER_TABLE).limit(5).withColumn(
    "loyalty_tier",
    F.when(F.col("loyalty_tier") == "Bronze", "Silver")
     .when(F.col("loyalty_tier") == "Silver", "Gold")
     .otherwise(F.col("loyalty_tier"))
)

print("Simulated updates:")
display(updates)

---
### 🎯 EXERCISE 5: Implement SCD Type 2 MERGE
**Your task**: Update the SCD2 table using Delta's MERGE operation.
**Two-step process:**

**Step 1**: Close existing current records that changed
- Match on `customer_id` and `is_current = true`
- Check if `loyalty_tier` changed
- Update: `effective_end_date = current_date()`, `is_current = false`

**Step 2**: Insert new versions
- Find which customers changed (join with closed records)
- Append new rows with updated data

**Pattern:**
```python
target_table.alias("target").merge(
updates.alias("updates"),
"target.customer_id = updates.customer_id AND target.is_current = true"
).whenMatchedUpdate(
condition="target.loyalty_tier != updates.loyalty_tier",
set={...}
).execute()
```
**Hint**: Use `DeltaTable.forName()` to get the target table.

In [0]:
# TODO: Implement SCD Type 2 MERGE

# Get target table
target_table = DeltaTable.forName(spark, CUSTOMERS_SCD2_TABLE)

# Prepare updates with SCD2 columns
updates_prepared = updates.select(
    F.col("customer_id"),
    F.col("first_name"),
    F.col("last_name"),
    F.col("email"),
    F.col("phone"),
    F.col("location"),
    F.col("loyalty_tier"),
    F.col("account_created_date").cast("date"),
    F.current_date().alias("effective_start_date"),
    F.lit(None).cast("date").alias("effective_end_date"),
    F.lit(True).alias("is_current"),
    F.lit(2).alias("version")
)

# TODO: Step 1 - Close existing records that changed
target_table.alias("target").merge(
    # TODO: Fill in merge logic
    
    
).whenMatchedUpdate(
    # TODO: Fill in update condition and set
    
    
).execute()

print("Closed changed records")

---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: SCD Type 2 MERGE (Step 1)

# target_table = DeltaTable.forName(spark, CUSTOMERS_SCD2_TABLE)

# updates_prepared = updates.select(
#     F.col("customer_id"),
#     F.col("first_name"),
#     F.col("last_name"),
#     F.col("email"),
#     F.col("phone"),
#     F.col("location"),
#     F.col("loyalty_tier"),
#     F.col("account_created_date").cast("date"),
#     F.current_date().alias("effective_start_date"),
#     F.lit(None).cast("date").alias("effective_end_date"),
#     F.lit(True).alias("is_current"),
#     F.lit(2).alias("version")
# )

# target_table.alias("target").merge(
#     updates_prepared.alias("updates"),
#     "target.customer_id = updates.customer_id AND target.is_current = true"
# ).whenMatchedUpdate(
#     condition="target.loyalty_tier != updates.loyalty_tier",
#     set={
#         "effective_end_date": F.current_date(),
#         "is_current": "false"
#     }
# ).execute()

# print("✅ Closed changed records")

In [0]:
# Step 2: Insert new versions for changed records
changed_customers = updates_prepared.alias("updates").join(
    spark.table(CUSTOMERS_SCD2_TABLE)
        .filter(F.col("is_current") == False)
        .filter(F.col("effective_end_date") == F.current_date())
        .select("customer_id")
        .alias("changed"),
    "customer_id",
    "inner"
)

if changed_customers.count() > 0:
    changed_customers.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(CUSTOMERS_SCD2_TABLE)
    
    print(f"✅ Inserted {changed_customers.count()} new versions")
else:
    print("No changes detected")

In [0]:
# Verify SCD2 changes - show customers with multiple versions
display(
    spark.table(CUSTOMERS_SCD2_TABLE)
        .groupBy("customer_id")
        .agg(F.count("*").alias("version_count"))
        .filter("version_count > 1")
        .join(
            spark.table(CUSTOMERS_SCD2_TABLE),
            "customer_id"
        )
        .orderBy("customer_id", "effective_start_date")
)

## Section 6: Data Quality Monitoring
Production pipelines need continuous quality monitoring.

In [0]:
# Customer quality metrics
customer_quality = spark.table(CUSTOMERS_SILVER_TABLE).select(
    F.lit("Customers").alias("entity"),
    F.count("*").alias("total_records"),
    F.sum(F.when(F.col("is_valid_email"), 1).otherwise(0)).alias("valid_emails"),
    F.sum(F.when(F.col("is_valid_phone"), 1).otherwise(0)).alias("valid_phones"),
    F.avg("data_quality_score").alias("avg_quality_score")
)

display(customer_quality)

In [0]:
# Product quality metrics
product_quality = spark.table(PRODUCTS_SILVER_TABLE).select(
    F.lit("Products").alias("entity"),
    F.count("*").alias("total_records"),
    F.sum(F.when(F.col("is_valid_price"), 1).otherwise(0)).alias("valid_prices"),
    F.sum(F.when(F.col("is_valid_cost"), 1).otherwise(0)).alias("valid_costs"),
    F.avg("profit_margin").alias("avg_profit_margin")
)

display(product_quality)

In [0]:
# Combined quality dashboard
display(spark.sql(f"""
    SELECT 
        'Bronze → Silver Pipeline' as pipeline,
        current_timestamp() as check_time,
        'customers' as table_name,
        COUNT(*) as record_count,
        SUM(CASE WHEN is_valid_email THEN 1 ELSE 0 END) as quality_checks_passed
    FROM {CUSTOMERS_SILVER_TABLE}
    
    UNION ALL
    
    SELECT 
        'Bronze → Silver Pipeline',
        current_timestamp(),
        'products',
        COUNT(*),
        SUM(CASE WHEN is_valid_price THEN 1 ELSE 0 END)
    FROM {PRODUCTS_SILVER_TABLE}
    
    UNION ALL
    
    SELECT 
        'Bronze → Silver Pipeline',
        current_timestamp(),
        'sales',
        COUNT(*),
        SUM(CASE WHEN is_valid_sale THEN 1 ELSE 0 END)
    FROM {SALES_SILVER_TABLE}
"""))

## Section 7: Summary and Checkpoint
### 🎯 Key Concepts Covered
**1. Medallion Architecture**
- Bronze: Raw ingestion with no transformations
- Silver: Cleaned, validated, conformed data
- Gold: Business-level aggregations (next notebook)

**2. Data Cleaning Techniques**
- Deduplication with window functions
- Standardization (case, trim, format)
- Validation (regex, ranges, business rules)
- Quality scoring

**3. Stream Processing**
- Stream-static joins
- Enrichment with dimension tables
- Calculated fields
- Checkpointing for streaming writes

**4. SCD Type 2**
- Effective date ranges
- Current record flags
- MERGE pattern for updates
- Historical tracking

**5. Data Quality**
- Validation flags
- Quality metrics
- Monitoring dashboards

### ✅ Exam Checklist
Can you:
- [ ] Implement deduplication with window functions?
- [ ] Apply standardization transformations?
- [ ] Join streaming and static dataframes?
- [ ] Write MERGE for SCD Type 2?
- [ ] Create validation logic?
- [ ] Explain Bronze/Silver/Gold layers?
### 📚 Next Steps
**Notebook 04** covers:
- Silver to Gold transformations
- Window functions for analytics
- Customer lifetime value
- Business-level aggregations
---
**🎉 Notebook Complete!**
The Silver layer is populated with clean, validated data ready for business analytics. Proceed to Notebook 04 for Gold layer transformations.